# 03. Policy Pipeline — Authorize every tool call, fail closed

A modern agent runtime spends most of its security budget at one moment: the instant
right before a tool fires. The model has decided what to do. The tool harness has
its arguments ready. Network sockets are warm. **This is the only place where
"don't do that" is still cheap.**

`arctrust.policy` is the engine that lives at that gate. It evaluates every
proposed tool invocation against an ordered stack of policy layers, returns
`allow` or `deny`, and emits a structured audit record. It is short-circuiting
(first DENY wins), fail-closed (an exception in any layer is treated as DENY),
and tier-aware (federal stacks seven layers, enterprise six, personal two).

You will learn:

- The shapes that flow through the pipeline: `ToolCall`, `PolicyContext`, `Decision`
- How `PolicyLayer` is just a Protocol — anyone can write one
- How to compose layers into a `PolicyPipeline` and how the `build_pipeline` factory works
- Why **every call must be authenticated** — `IdentityLayer` runs first at every tier,
  so a `ToolCall` must be signed by an `AgentIdentity` before any other layer even sees it
- Why **first-DENY-wins** is the right default and how to demonstrate it
- Why **fail-closed** is the right default — and how to prove it with an exploding layer
- How the personal / enterprise / federal tier matrix assembles a different layer stack
  via `build_pipeline`
- Where this fits in the agent loop (`arcagent`)

This entire notebook runs without a network call, a filesystem, or a real API key — but
it does mint real Ed25519 keypairs (via `AgentIdentity.generate(...)`) to sign the tool
calls the identity layer now demands of every call, at every tier.</cell id="0b59c377">


## 1. Setup

The boilerplate cell below makes every `packages/<pkg>/src` and `packages/<pkg>/tests` directory importable. Run it once and forget it for the rest of the notebook.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Pull in every public symbol we need from `arctrust.policy`. Note that the policy engine has no required runtime — no DB, no key file, no network. Just types and a tiny async evaluator.

In [2]:
from arctrust import AgentIdentity
from arctrust.policy import (
    Decision,
    PolicyContext,
    PolicyLayer,
    PolicyPipeline,
    ToolCall,
    build_pipeline,
    sign_call,
)

# Sanity check: these are the symbols this notebook is responsible for.
public_api = {
    "PolicyContext": PolicyContext,
    "PolicyPipeline": PolicyPipeline,
    "ToolCall": ToolCall,
    "build_pipeline": build_pipeline,
    "sign_call": sign_call,
    "AgentIdentity": AgentIdentity,
}
for name, obj in public_api.items():
    print(f"{name:18s} -> {obj}")

PolicyContext      -> <class 'arctrust.policy.PolicyContext'>
PolicyPipeline     -> <class 'arctrust.policy.PolicyPipeline'>
ToolCall           -> <class 'arctrust.policy.ToolCall'>
build_pipeline     -> <function build_pipeline at 0x10ca84f40>
sign_call          -> <function sign_call at 0x10c9a0400>
AgentIdentity      -> <class 'arctrust.identity.AgentIdentity'>


All six are importable. We also imported `Decision` and `PolicyLayer` because every example below produces or implements them, and `AgentIdentity` + `sign_call` because — as you're about to see — every call now has to be cryptographically signed before the pipeline will look at it.

## 2. Why a policy pipeline

The agent loop is a long, mostly-deterministic computation that ends each turn
with one or more **tool calls** the model wants to dispatch. Without a policy
gate, the only thing standing between the model and the tool's side effects is
hope.

That fails three ways:

1. **Least privilege.** Most agents only need a tiny slice of the available
   tool surface. A summarizer doesn't need `delete_file`. A search agent
   doesn't need `transfer_funds`. Without an enforced allowlist, "the agent
   could do anything" is functionally true.

2. **Defense in depth.** A model that has been jailbroken or prompt-injected
   does not know it has been compromised. The runtime needs an authority that
   sits *outside* the model's reasoning loop and can refuse to dispatch.

3. **Federal context.** SPEC-017 R-010 to R-018 mandate a layered, fail-closed,
   tamper-evident decision boundary on every tool call. The pipeline IS that
   boundary; everything else (audit, signing, identity) attaches to it.

The pipeline is intentionally **boring**: ordered list of layers, first DENY
wins, exceptions become DENY, every decision is a structured `Decision` object
with the layer / rule / reason / input hash. Boring is the point — federal
auditors don't want clever, they want predictable.

In [3]:
# A pipeline is just an ordered sequence of layers. Empty pipeline allows
# vacuously — the policy says nothing, so nothing is denied.
empty = PolicyPipeline(layers=[])
print("layers in empty pipeline:", len(empty.layers))

layers in empty pipeline: 0


Vacuous-allow is the right default for an *empty* policy. The minute we add even one layer, the pipeline gets opinions.

## 3. Building blocks — `ToolCall`, `PolicyContext`, and signing

Every evaluation needs two pieces of input:

| Type            | What it carries                                                                          |
|-----------------|------------------------------------------------------------------------------------------|
| `ToolCall`      | The thing the agent wants to do — tool name, arguments, who is asking, what session, and (as of SPEC-038) the caller's Ed25519 signature over that content. |
| `PolicyContext` | The runtime context — deployment tier, policy bundle version, age of the bundle, and optionally clearance/provider-usage/team-scope state the newer layers consult. |

`ToolCall` and `PolicyContext` are Pydantic `frozen=True` models. They are
immutable by construction, which makes them safe to pass around and cache.</cell id="fbeab702">


In [4]:
call = ToolCall(
    tool_name="read_file",
    arguments={"path": "/etc/hostname"},
    agent_did="did:arc:demo:summarizer/0123abcd",
    session_id="sess-001",
    classification="UNCLASSIFIED",
)
print("tool_name:     ", call.tool_name)
print("agent_did:     ", call.agent_did)
print("classification:", call.classification)
print("session_id:    ", call.session_id)
print("parent_call_id:", call.parent_call_id)  # None unless this is a delegated call

tool_name:      read_file
agent_did:      did:arc:demo:summarizer/0123abcd
classification: UNCLASSIFIED
session_id:     sess-001
parent_call_id: None


`ToolCall` carries an optional `parent_call_id`. When a parent agent delegates to a child, the child's tool calls reference the parent so classification propagation can be audited.

In [5]:
# Frozen — try to mutate and watch it complain.
try:
    call.tool_name = "delete_file"  # type: ignore[misc]
except Exception as e:
    print(f"frozen model refused mutation: {type(e).__name__}: {e}")

frozen model refused mutation: ValidationError: 1 validation error for ToolCall
tool_name
  Instance is frozen [type=frozen_instance, input_value='delete_file', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/frozen_instance


Now the `PolicyContext` — what tier are we, and how stale is the policy bundle? Bundle age matters for restricted mode, where we deny everything except a small safe-set when the bundle hasn't refreshed in a while.

In [6]:
ctx = PolicyContext(
    tier="personal",
    policy_version="2026-05-07.r1",
    bundle_age_seconds=12.5,
)
print("tier:               ", ctx.tier)
print("policy_version:     ", ctx.policy_version)
print("bundle_age_seconds: ", ctx.bundle_age_seconds)

tier:                personal
policy_version:      2026-05-07.r1
bundle_age_seconds:  12.5


A `ToolCall` built with no `public_key`/`signature` (like the one above) is
**unauthenticated**. `arctrust.policy.verify_call(call)` — the predicate the
identity layer runs — returns `False` for it: no signature to check, so no
proof the caller holds the private key for the claimed `agent_did`. Once we
start running calls through `build_pipeline(...)`-assembled pipelines in
section 5, that matters, because `IdentityLayer` runs FIRST at *every* tier
(SPEC-038 / ADR-019) — personal included. An unsigned call never even reaches
`global`/`agent`/`sandbox`; it dies at the door.

`sign_call(call, identity)` fixes that: it stamps the call with the identity's
public key, overwrites `agent_did` to the signer's own DID (so you can't sign
content while claiming someone else's identity), and attaches an Ed25519
signature over the call's canonical bytes.

In [7]:
from arctrust.policy import verify_call

# Mint a real agent identity — a fresh Ed25519 keypair + a derived DID.
summarizer = AgentIdentity.generate(org="demo", agent_type="summarizer")
print("signer DID:", summarizer.did)

unsigned = ToolCall(
    tool_name="read_file",
    arguments={"path": "/etc/hostname"},
    agent_did=summarizer.did,
    session_id="sess-001",
    classification="UNCLASSIFIED",
)
signed = sign_call(unsigned, summarizer)

print("unsigned verify_call():", verify_call(unsigned))
print("signed   verify_call():", verify_call(signed))
print("signed has public_key:", signed.public_key is not None)
print("signed has signature: ", signed.signature is not None)

signer DID: did:arc:demo:summarizer/b47e6b46
unsigned verify_call(): False
signed   verify_call(): True
signed has public_key: True
signed has signature:  True


Two things to notice:

- **Tampering breaks the signature.** `signing_bytes()` covers `tool_name`,
  `arguments`, `agent_did`, `session_id`, `classification`, `parent_call_id` —
  everything except the `public_key`/`signature` pair itself (they can't
  authenticate themselves). Change any of those fields after signing and
  `verify_call` goes back to `False`.
- **The DID must match the key.** `sign_call` derives `agent_did` from the
  identity that signs, so a call can never claim to be from `alice` while
  actually being signed by `bob`'s key — `did_matches_pubkey` enforces the
  binding.

We'll build a proper `AgentIdentity` + `sign_call` helper below and reuse it
for every `build_pipeline(...)`-based demo for the rest of this notebook.</cell id="b45c6387">


In [8]:
# Tamper with a signed call after the fact — verify_call must catch it.
tampered = signed.model_copy(update={"arguments": {"path": "/etc/shadow"}})
print("tampered call verify_call():", verify_call(tampered))

# A DID that doesn't match the presented key is caught too — sign as bob,
# then splice in alice's DID before sending.
bob = AgentIdentity.generate(org="demo", agent_type="planner")
bob_call = sign_call(
    ToolCall(
        tool_name="read_file",
        arguments={},
        agent_did=bob.did,
        session_id="sess-001",
        classification="UNCLASSIFIED",
    ),
    bob,
)
spoofed = bob_call.model_copy(update={"agent_did": summarizer.did})
print("DID-spoofed call verify_call():", verify_call(spoofed))

tampered call verify_call(): False
DID-spoofed call verify_call(): False


## 4. Single-layer evaluation — one rule allowing or denying a call

Before composing layers, build the smallest possible pipeline: one layer, one
rule, one tool call.

`PolicyLayer` is a `runtime_checkable` `Protocol`. Anything with a `name`
attribute and an `async evaluate(call, ctx) -> Decision` satisfies it. No
inheritance, no registration — just shape.

In [9]:
import time


class WriteOnlyDeny:
    """Trivial layer: deny anything that looks like a write."""

    name = "write_only_deny"

    async def evaluate(self, call: ToolCall, ctx: PolicyContext) -> Decision:
        is_write = call.tool_name.startswith(("write_", "delete_", "send_"))
        now_us = int(time.monotonic() * 1_000_000)
        if is_write:
            return Decision.deny(
                layer=self.name,
                rule_id="no_writes",
                reason=f"Tool {call.tool_name!r} performs a write; this layer is read-only.",
                input_hash="demo",
                evaluated_at_us=now_us,
            )
        return Decision.allow(input_hash="demo", evaluated_at_us=now_us)


# Confirm the Protocol is satisfied.
layer = WriteOnlyDeny()
assert isinstance(layer, PolicyLayer)
print("WriteOnlyDeny satisfies PolicyLayer protocol:", isinstance(layer, PolicyLayer))

WriteOnlyDeny satisfies PolicyLayer protocol: True


Now build a one-layer pipeline and evaluate two calls — a read (allowed) and a write (denied).

In [10]:
pipeline = PolicyPipeline(layers=[WriteOnlyDeny()])

read_call = ToolCall(
    tool_name="read_file",
    arguments={"path": "/etc/hostname"},
    agent_did="did:arc:demo:summarizer/0123abcd",
    session_id="sess-001",
    classification="UNCLASSIFIED",
)
write_call = ToolCall(
    tool_name="delete_file",
    arguments={"path": "/etc/passwd"},
    agent_did="did:arc:demo:summarizer/0123abcd",
    session_id="sess-001",
    classification="UNCLASSIFIED",
)

ctx = PolicyContext(tier="personal", policy_version="demo", bundle_age_seconds=0.0)

read_decision = await pipeline.evaluate(read_call, ctx)
write_decision = await pipeline.evaluate(write_call, ctx)

print("read_file   ->", read_decision.outcome)
print(
    "delete_file ->",
    write_decision.outcome,
    "| layer:",
    write_decision.layer,
    "| rule:",
    write_decision.rule_id,
    "| reason:",
    write_decision.reason,
)

read_file   -> allow
delete_file -> deny | layer: write_only_deny | rule: no_writes | reason: Tool 'delete_file' performs a write; this layer is read-only.


Three things in the deny output that are not optional:

1. **`layer`** — which layer made the decision (`write_only_deny`).
2. **`rule_id`** — which rule inside that layer (`no_writes`).
3. **`reason`** — human-readable explanation suitable for an audit log.

SPEC-017 R-014 calls these the three structured answers an auditor will demand: which layer, which rule, what inputs. Every `Decision.deny(...)` carries them.

In [11]:
# A Decision is itself frozen. Once made, it cannot be amended.
try:
    write_decision.outcome = "allow"  # type: ignore[misc]
except Exception as e:
    print(f"frozen decision refused mutation: {type(e).__name__}")

frozen decision refused mutation: ValidationError


## 5. Pipeline composition — `build_pipeline` and the tier matrix

Hand-rolling layers is fine for tests and bespoke deployments. For the
common case you want `build_pipeline()`, which assembles the right layer
set for a given tier and wires the standard built-ins:

| Tier        | Layer order                                                                      | Count |
|-------------|-----------------------------------------------------------------------------------|-------|
| personal    | `identity` -> `global`                                                            | 2     |
| enterprise  | `identity` -> `global` -> `classification` -> `provider` -> `agent` -> `sandbox`  | 6     |
| federal     | `identity` -> `global` -> `classification` -> `provider` -> `agent` -> `team` -> `sandbox` | 7 |

`identity` runs first at every tier — that's the signing requirement from
section 3. `classification` is the SPEC-038 no-read-up gate (Bell-LaPadula
"no read up" over `PolicyContext.clearance`); it's a no-op ALLOW when no
clearance context is attached, unless the deployment turned on enforcement
(federal always does).

The factory takes optional `global_deny_rules` (tool-name -> reason),
`agent_allowlists` (DID -> set of tools), `agent_registry` (DID -> pubkey,
consulted by `identity` when `require_registered` — enterprise/federal),
`provider_limits`, `team_roles`, `cache_ttl_seconds`, `max_bundle_age_seconds`,
`safe_set`, `shadow`, and `audit_sink`.

Layer ordering is deliberate. Cheapest, broadest checks run first
(`identity`, `global`); narrowest, most-expensive last (`sandbox`). When an
early layer denies, we never reach the later ones — saving compute and
minimizing what an adversary can probe by enumerating denials.</cell id="4e734b04">


In [12]:
personal_pipeline = build_pipeline(tier="personal")
enterprise_pipeline = build_pipeline(tier="enterprise")
federal_pipeline = build_pipeline(tier="federal")

for label, pipeline in [
    ("personal", personal_pipeline),
    ("enterprise", enterprise_pipeline),
    ("federal", federal_pipeline),
]:
    layer_names = [layer.name for layer in pipeline.layers]
    print(f"{label:10s} ({len(layer_names)} layers): {layer_names}")

personal   (2 layers): ['identity', 'global']
enterprise (6 layers): ['identity', 'global', 'classification', 'provider', 'agent', 'sandbox']
federal    (7 layers): ['identity', 'global', 'classification', 'provider', 'agent', 'team', 'sandbox']


The pipeline exposes its layers via `.layers` for introspection. This is a *read-only copy* — mutating the returned list will not mutate the pipeline.

In [13]:
# Build a personal pipeline with a tenant-wide denylist.
pipeline = build_pipeline(
    tier="personal",
    global_deny_rules={
        "rm_rf": "Recursive delete is never allowed in this tenant.",
        "transfer_funds": "Financial primitives require an enterprise tier.",
        "exfiltrate_data": "Data exfiltration tools are categorically forbidden.",
    },
)

ctx = PolicyContext(tier="personal", policy_version="demo", bundle_age_seconds=0.0)


async def show_global_denylist() -> None:
    for tool in ("read_file", "rm_rf", "transfer_funds", "search_web"):
        call = sign_call(
            ToolCall(
                tool_name=tool,
                arguments={},
                agent_did=summarizer.did,
                session_id="sess-002",
                classification="UNCLASSIFIED",
            ),
            summarizer,
        )
        decision = await pipeline.evaluate(call, ctx)
        flag = "BLOCK" if decision.is_deny() else " ok  "
        detail = f"({decision.layer}/{decision.rule_id})" if decision.is_deny() else ""
        print(f"  {flag}  {tool:18s}  {detail}")


await show_global_denylist()

   ok    read_file           
  BLOCK  rm_rf               (global/global.denylist)
  BLOCK  transfer_funds      (global/global.denylist)
   ok    search_web          


`global` is enough for personal-tier deployments. For enterprise / federal, add per-agent allowlists so that even tools the tenant *broadly* permits can be locked down for a specific agent.

In [14]:
# Enterprise pipeline: tenant allows everything except `rm_rf`, but the
# summarizer agent specifically may only call read_file and search_web.
#
# Enterprise/federal turn on `require_registered` in the identity layer, so
# the agent must ALSO appear in `agent_registry` (DID -> pubkey) or it gets
# denied at `identity` before the `agent` allowlist even runs.
ent_pipeline = build_pipeline(
    tier="enterprise",
    global_deny_rules={"rm_rf": "Recursive delete is never allowed."},
    agent_registry={summarizer.did: summarizer.public_key},
    agent_allowlists={
        summarizer.did: {"read_file", "search_web"},
    },
)

ctx = PolicyContext(tier="enterprise", policy_version="demo", bundle_age_seconds=0.0)


async def show_enterprise_stack() -> None:
    for tool in ("read_file", "write_file", "rm_rf", "search_web"):
        call = sign_call(
            ToolCall(
                tool_name=tool,
                arguments={},
                agent_did=summarizer.did,
                session_id="sess-003",
                classification="UNCLASSIFIED",
            ),
            summarizer,
        )
        decision = await ent_pipeline.evaluate(call, ctx)
        flag = "BLOCK" if decision.is_deny() else " ok  "
        layer = decision.layer or ""
        print(f"  {flag}  {tool:12s}  layer={layer}")


await show_enterprise_stack()

   ok    read_file     layer=
  BLOCK  write_file    layer=agent
  BLOCK  rm_rf         layer=global
   ok    search_web    layer=


Two layers each had something to say:

- `global` denied `rm_rf` (tenant-wide rule).
- `agent` denied `write_file` (not on the summarizer's allowlist).

`read_file` and `search_web` ran the full stack and emerged allowed. The fact that the agent layer comes *after* global means that a global denylist can't be circumvented by widening an agent's allowlist — global wins first.

## 6. First-DENY-wins — explicit demo

Pipeline ordering matters because the pipeline **short-circuits on the first
DENY**. A later layer that would have allowed the call never even runs. This
is more than an optimization — it ensures every denial is attributable to a
single, specific layer / rule pair, which is what auditors care about.

Build a deliberately conflicting pair of layers — one denies, the next allows
— and watch the pipeline pick DENY without consulting the second layer.

In [15]:
import time


class TrackingAllow:
    """Tracks whether it was called. Always allows."""

    name = "tracking_allow"

    def __init__(self) -> None:
        self.called = False

    async def evaluate(self, call: ToolCall, ctx: PolicyContext) -> Decision:
        self.called = True
        return Decision.allow(
            input_hash="demo",
            evaluated_at_us=int(time.monotonic() * 1_000_000),
        )


class TrackingDeny:
    """Tracks whether it was called. Always denies."""

    name = "tracking_deny"

    def __init__(self) -> None:
        self.called = False

    async def evaluate(self, call: ToolCall, ctx: PolicyContext) -> Decision:
        self.called = True
        return Decision.deny(
            layer=self.name,
            rule_id="always_deny",
            reason="Configured to deny everything.",
            input_hash="demo",
            evaluated_at_us=int(time.monotonic() * 1_000_000),
        )


early_deny = TrackingDeny()
late_allow = TrackingAllow()

pipeline = PolicyPipeline(layers=[early_deny, late_allow])

call = ToolCall(
    tool_name="any_tool",
    arguments={},
    agent_did="did:arc:demo:any/0123abcd",
    session_id="sess-006",
    classification="UNCLASSIFIED",
)
ctx = PolicyContext(tier="personal", policy_version="demo", bundle_age_seconds=0.0)

decision = await pipeline.evaluate(call, ctx)
print("outcome:                  ", decision.outcome)
print("layer that decided:       ", decision.layer)
print("first (deny) layer called:", early_deny.called)
print("second (allow) layer called:", late_allow.called, "  <-- short-circuited away")

outcome:                   deny
layer that decided:        tracking_deny
first (deny) layer called: True
second (allow) layer called: False   <-- short-circuited away


`late_allow.called == False` is the proof. The pipeline saw the DENY from the first layer and stopped — the allow layer never got a chance to overrule it. **No layer can recover from another layer's DENY.**

In [16]:
# Flip the order. Now the allow layer runs first, sees nothing wrong,
# and the deny layer fires on the second pass — still DENY.
allow_first = TrackingAllow()
deny_after = TrackingDeny()

pipeline2 = PolicyPipeline(layers=[allow_first, deny_after])
decision2 = await pipeline2.evaluate(call, ctx)
print("outcome:                ", decision2.outcome)
print("allow_first.called:     ", allow_first.called)
print("deny_after.called:      ", deny_after.called)

outcome:                 deny
allow_first.called:      True
deny_after.called:       True


This shows the pipeline does keep walking past `allow` outcomes — it only short-circuits on `deny`. The mental model: every layer is consulted in order until one says no. If none says no, the result is allow.

## 7. Tier-aware decisions — same call, different tier, different verdict

The same `ToolCall` can yield different decisions depending on the tier the
pipeline was built for. This is how Arc supports the personal / enterprise
/ federal tier matrix without three forks of the codebase.

Watch a call hit each tier in turn.

In [17]:
from arctrust.classification import Classification
from arctrust.policy import ClearanceContext

# Federal ALSO forces classification enforcement unconditionally (ADR-019),
# so a federal PolicyContext with no clearance state fails closed at the
# `classification` layer — before it ever reaches `agent`. Supply a matching
# UNCLASSIFIED/UNCLASSIFIED clearance so classification allows through and we
# can actually observe the agent-layer allowlist decision at federal too.
_federal_clearance = ClearanceContext(
    caller_clearance=Classification.UNCLASSIFIED,
    resource_classification=Classification.UNCLASSIFIED,
)


async def evaluate_across_tiers(tool_name: str, identity: AgentIdentity) -> None:
    for tier in ("personal", "enterprise", "federal"):
        pipeline = build_pipeline(
            tier=tier,
            global_deny_rules={"rm_rf": "Tenant-wide block."},
            # Every agent evaluated here must be admitted at enterprise/federal
            # (identity.require_registered), regardless of whether it also has
            # an agent-layer allowlist entry.
            agent_registry={identity.did: identity.public_key},
            # Only the summarizer has an allowlist entry (restricted to read_file).
            # Any other agent DID is opt-in unconstrained at the agent layer.
            agent_allowlists={summarizer.did: {"read_file"}},
        )
        call = sign_call(
            ToolCall(
                tool_name=tool_name,
                arguments={},
                agent_did=identity.did,
                session_id="sess-007",
                classification="UNCLASSIFIED",
            ),
            identity,
        )
        ctx = PolicyContext(
            tier=tier,
            policy_version="demo",
            bundle_age_seconds=0.0,
            clearance=_federal_clearance if tier == "federal" else None,
        )
        decision = await pipeline.evaluate(call, ctx)
        verdict = decision.outcome.upper()
        layer = decision.layer or "-"
        print(f"  {tier:10s} -> {verdict:5s}  (layer: {layer})")


planner = AgentIdentity.generate(org="demo", agent_type="planner")

print("call: write_file from summarizer DID")
await evaluate_across_tiers("write_file", summarizer)

print()
print("call: write_file from a different agent (no allowlist entry)")
await evaluate_across_tiers("write_file", planner)

call: write_file from summarizer DID
  personal   -> ALLOW  (layer: -)
  enterprise -> DENY   (layer: agent)
  federal    -> DENY   (layer: agent)

call: write_file from a different agent (no allowlist entry)
  personal   -> ALLOW  (layer: -)
  enterprise -> ALLOW  (layer: -)
  federal    -> ALLOW  (layer: -)


Four things to notice:

1. **Personal tier** has no agent layer, so the summarizer's restricted allowlist
   does not apply — `write_file` slips through. This is correct: personal
   deployments are meant to be light-touch.

2. **Enterprise and federal tiers** both run the agent layer. The summarizer is
   pinned to `read_file` only, so `write_file` is denied at the `agent` layer
   — but only once we supply a matching `ClearanceContext` for federal.
   Without one, federal denies everything at the earlier `classification`
   layer (`classification.state_missing`) — try dropping the `clearance=...`
   line above and rerun to see it. That's the SPEC-038 no-read-up floor:
   federal fails closed on missing clearance state; personal/enterprise don't
   enforce it unless the deployment explicitly opts in.

3. **Different agent, same call.** The `planner` DID has *no* allowlist entry —
   it is unconstrained at the agent layer. So `write_file` is allowed for the
   planner across all three tiers (once identity + classification are
   satisfied). Allowlist is opt-in: if you don't list an agent, that layer
   doesn't restrict it.

4. **`agent_registry` is not optional at enterprise/federal.** Both identities
   above were added to the registry passed into `build_pipeline`; omit that
   and every call denies at `identity` (`identity.not_admitted`) before any
   of this even runs.</cell id="05641f79">


In [18]:
# There is no separate TierConfig object anymore, and arctrust never sized a
# tool-execution semaphore — it only ever decided allow/deny per call. Bounding
# concurrent tool dispatch is a plain arcrun concern: ParallelDispatcher(max_parallel=...)
# in arcrun.parallel_dispatch, a deployment-set constructor parameter (default 10),
# not a tier-derived FIPS floor.
from arcrun.parallel_dispatch import ParallelDispatcher

default_dispatcher = ParallelDispatcher()
print("arcrun ParallelDispatcher default max_parallel:", default_dispatcher._max_parallel)

arcrun ParallelDispatcher default max_parallel: 10


There used to be a `TierConfig.max_parallel_tools` FIPS cap read by `arcagent`'s dispatch
layer; that object no longer exists. Concurrency limiting lives entirely in `arcrun` now,
as a plain constructor parameter with no tier coupling — the separation of concerns is
still intentional (`arctrust` decides allow/deny; `arcrun` decides how many calls run at
once), it's just no longer expressed through a shared `TierConfig` type.

## 8. Fail-closed semantics — exceptions become DENY

Layers MUST NOT raise. But software has bugs. A poorly-written layer might
hit a `KeyError` because someone changed an upstream schema, or run out of
memory, or just deadlock. SPEC-017 R-012 says: **any exception in a layer is
caught by the pipeline and converted to DENY.** It is never allowed to
propagate.

This is "fail-closed" — when the policy engine cannot determine an answer,
the safe answer is no. The alternative ("fail-open" — allow on failure)
would be catastrophic: any adversary who could cause the pipeline to throw
would have effectively bypassed every policy.

Demonstrate it with a layer that simply raises.

In [19]:
class ExplodingLayer:
    """Always raises. The pipeline must catch this and DENY."""

    name = "exploder"

    async def evaluate(self, call: ToolCall, ctx: PolicyContext) -> Decision:
        raise RuntimeError("simulated upstream failure: trust store offline")


pipeline = PolicyPipeline(layers=[ExplodingLayer()])

call = ToolCall(
    tool_name="any_tool",
    arguments={},
    agent_did="did:arc:demo:any/0123abcd",
    session_id="sess-008",
    classification="UNCLASSIFIED",
)
ctx = PolicyContext(tier="personal", policy_version="demo", bundle_age_seconds=0.0)

# This call MUST NOT raise — the pipeline catches the layer's RuntimeError
# and converts it to a structured DENY decision.
decision = await pipeline.evaluate(call, ctx)

print("outcome:  ", decision.outcome)
print("layer:    ", decision.layer)
print("rule_id:  ", decision.rule_id)
print("reason:   ", decision.reason)

Policy layer 'exploder' raised — failing closed (R-012)
Traceback (most recent call last):
  File "/Users/joshschultz/Projects/arc/.claude/worktrees/agent-af64d3fc2a27f5372/packages/arctrust/src/arctrust/policy.py", line 981, in _eval_layer
    return await layer.evaluate(call, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/ipykernel_63075/2185369068.py", line 7, in evaluate
    raise RuntimeError("simulated upstream failure: trust store offline")
RuntimeError: simulated upstream failure: trust store offline


outcome:   deny
layer:     exploder
rule_id:   layer_error
reason:    RuntimeError: simulated upstream failure: trust store offline


The `rule_id` is the well-known sentinel `layer_error`. The `reason` includes
the exception type and message — useful for postmortems, useless for an
attacker (no stack frames, no internals). The `layer` is the offending
layer's name, so an SRE can immediately go look at it.

What about a layer that allows, *then* a later layer that explodes? The
exception still wins — and the call is denied.

In [20]:
class AlwaysAllow:
    name = "always_allow"

    async def evaluate(self, call: ToolCall, ctx: PolicyContext) -> Decision:
        return Decision.allow(
            input_hash="demo",
            evaluated_at_us=int(time.monotonic() * 1_000_000),
        )


pipeline = PolicyPipeline(layers=[AlwaysAllow(), ExplodingLayer()])
decision = await pipeline.evaluate(call, ctx)

print("outcome:", decision.outcome, "| layer:", decision.layer, "| rule:", decision.rule_id)

Policy layer 'exploder' raised — failing closed (R-012)
Traceback (most recent call last):
  File "/Users/joshschultz/Projects/arc/.claude/worktrees/agent-af64d3fc2a27f5372/packages/arctrust/src/arctrust/policy.py", line 981, in _eval_layer
    return await layer.evaluate(call, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/ipykernel_63075/2185369068.py", line 7, in evaluate
    raise RuntimeError("simulated upstream failure: trust store offline")
RuntimeError: simulated upstream failure: trust store offline


outcome: deny | layer: exploder | rule: layer_error


Even though the *first* layer allowed the call, the *second* raising means
the pipeline cannot honestly answer "allow" — and so it must answer "deny".
This is non-negotiable: the moment any layer becomes unreliable, the whole
pipeline degrades to deny.

If you ever find yourself thinking "but the call really should have been
allowed because the broken layer wasn't relevant", you have rediscovered the
exact reasoning an attacker would weaponize. The pipeline is right.

## 9. Wiring policy into the agent loop

Where does the pipeline actually get called?

In `arcagent`, every `ArcAgent` is constructed with a `PolicyPipeline` (or
the agent builds one via `build_pipeline()` from its tier config). When the
LLM produces tool-use blocks for a turn, the agent's tool dispatcher does
roughly this:

```python
# Pseudocode — see arcagent.core.agent for the real implementation.
for tool_use in response.tool_calls:
    call = ToolCall(
        tool_name=tool_use.name,
        arguments=tool_use.arguments,
        agent_did=self.identity.did,
        session_id=self.session_id,
        classification=self.classification,
        parent_call_id=parent.id if parent else None,
    )
    decision = await self.policy_pipeline.evaluate(call, self.policy_context)
    if decision.is_deny():
        # Emit audit event, raise ToolVetoedError. The model sees the
        # veto reason and decides whether to retry with a different tool
        # or escalate to a human.
        raise ToolVetoedError(decision)
    result = await self.tool_registry.dispatch(call)
```

Three things to internalize about this wiring:

- **The pipeline is the only authority.** There is no second-chance hook,
  no "can we run this anyway" override. If the pipeline denies, the tool
  doesn't run. Period.
- **The decision becomes part of the audit trail.** `arctrust.audit.emit`
  records every evaluation; sinks fan out to JSONL and the signed chain.
  See notebook 04 for that story.
- **Caching is invisible to the agent.** With `cache_ttl_seconds > 0`,
  identical calls in a short window return the same decision without
  re-running layers. The agent never sees the difference.

The simulated dispatch below mimics what `arcagent` does, end to end.

In [21]:
class ToolVetoedError(RuntimeError):
    """Raised inside the agent loop when the pipeline denies a tool call."""

    def __init__(self, decision: Decision) -> None:
        super().__init__(f"{decision.layer}/{decision.rule_id}: {decision.reason}")
        self.decision = decision


class FakeAgentDispatcher:
    """A caricature of arcagent.core.agent.ArcAgent's tool-dispatch path."""

    def __init__(
        self,
        *,
        tier: str,
        identity: AgentIdentity,
        session_id: str,
        deny_rules: dict[str, str] | None = None,
        agent_allowlists: dict[str, set[str]] | None = None,
    ) -> None:
        self._pipeline = build_pipeline(
            tier=tier,
            global_deny_rules=deny_rules or {},
            # Enterprise/federal require the caller to be an admitted agent.
            agent_registry={identity.did: identity.public_key},
            agent_allowlists=agent_allowlists or {},
        )
        self._ctx = PolicyContext(tier=tier, policy_version="demo", bundle_age_seconds=0.0)
        self._identity = identity
        self._session_id = session_id

    async def dispatch(self, tool_name: str, arguments: dict) -> str:
        call = sign_call(
            ToolCall(
                tool_name=tool_name,
                arguments=arguments,
                agent_did=self._identity.did,
                session_id=self._session_id,
                classification="UNCLASSIFIED",
            ),
            self._identity,
        )
        decision = await self._pipeline.evaluate(call, self._ctx)
        if decision.is_deny():
            raise ToolVetoedError(decision)
        return f"executed {tool_name} with {arguments}"


agent = FakeAgentDispatcher(
    tier="enterprise",
    identity=summarizer,
    session_id="sess-009",
    deny_rules={"rm_rf": "tenant-wide block"},
    agent_allowlists={summarizer.did: {"read_file", "search_web"}},
)


async def simulate_turn() -> None:
    requested = [
        ("read_file", {"path": "notes.md"}),
        ("write_file", {"path": "out.md", "data": "..."}),
        ("rm_rf", {"path": "."}),
        ("search_web", {"q": "policy pipelines"}),
    ]
    for tool, args in requested:
        try:
            result = await agent.dispatch(tool, args)
            print(f"  ok      {tool:12s}  {result}")
        except ToolVetoedError as e:
            d = e.decision
            print(f"  vetoed  {tool:12s}  layer={d.layer:8s} rule={d.rule_id}")


await simulate_turn()

  ok      read_file     executed read_file with {'path': 'notes.md'}
  vetoed  write_file    layer=agent    rule=agent.allowlist
  vetoed  rm_rf         layer=global   rule=global.denylist
  ok      search_web    executed search_web with {'q': 'policy pipelines'}


Inside one simulated agent turn, the model "asked for" four tools. The
pipeline allowed two and vetoed two — once at the `global` layer (`rm_rf`)
and once at the `agent` layer (`write_file`). The vetoes carry their layer
and rule, so the audit trail is precise and the model can decide what to
do next (retry with a different tool, escalate, give up).

For the full agent-side story see:

- `walkthroughs/arcagent/01-first-agent.ipynb` — agent construction
- `walkthroughs/arcagent/02-tool-integration.ipynb` — tool registry + transports
- `walkthroughs/arcagent/03-policy-and-modules.ipynb` — policy + module bus

## 10. Summary

You built and exercised the `arctrust` policy engine end to end.

**What you saw**

- `ToolCall` and `PolicyContext` are immutable inputs. `Decision` is the
  immutable output. All Pydantic frozen models.
- `IdentityLayer` runs first at every tier (ADR-019) — an unsigned `ToolCall`
  is denied before any other layer even sees it. `sign_call`/`verify_call` are
  the signing seam.
- `PolicyLayer` is just a Protocol — anything with a `name` and an async
  `evaluate` qualifies. Custom layers are trivial.
- `PolicyPipeline` runs layers in order, **first DENY wins**, and is
  **fail-closed** — any exception becomes DENY with `rule_id="layer_error"`.
- `build_pipeline(tier=...)` is the factory — personal gets 2 layers
  (`identity`, `global`), enterprise 6, federal 7 (adds `classification`,
  `provider`, `agent`, `team`, `sandbox`). There is no separate `TierConfig`
  object; the matrix is built directly by the factory.
- Concurrency limiting (how many tool calls run in parallel) moved out of
  `arctrust` entirely — it's a plain `arcrun.parallel_dispatch.ParallelDispatcher`
  parameter now, not a tier-derived cap.

**Public API exercised in this notebook**

| Symbol            | Where                             |
|-------------------|-----------------------------------|
| `PolicyContext`   | section 3, 4, 5, 6, 7, 8, 9       |
| `PolicyPipeline`  | section 4, 5, 6, 7, 8, 9          |
| `ToolCall`        | section 3, 4, 5, 6, 7, 8, 9       |
| `sign_call` / `verify_call` | section 3                |
| `build_pipeline`  | section 5, 7, 9                   |
| `Decision`        | section 4, 5, 6, 7, 8, 9 (output) |
| `PolicyLayer`     | section 4 (Protocol check)        |
| `arcrun.parallel_dispatch.ParallelDispatcher` | section 7 |

**Next**

- `walkthroughs/arctrust/04-audit-sinks.ipynb` — every decision above flows
  into an audit trail. That notebook covers `AuditEvent`, `AuditSink`,
  `NullSink`, `WormSink`, and `emit`. The `audit_sink` parameter on
  `PolicyPipeline` is the bridge between the two.
- `walkthroughs/arcagent/03-policy-and-modules.ipynb` — see the policy
  pipeline plugged into a real `ArcAgent` with module-bus integration.